# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [1]:
import asyncio
import json
import os
import time
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
from openai import AsyncOpenAI

load_dotenv()

# Make sure your OPENAI_API_KEY is set in the environment
#assert os.environ.get('OPENAI_API_KEY')
api_base = os.getenv("OPEN_API_BASE")
api_key  = os.getenv("OPEN_API_KEY")

client = AsyncOpenAI(api_key=api_key)

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [2]:
DATA_DIR = Path('./data')   # adjust if your folder layout differs

snippets = [
    json.loads(line) 
    for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() 
    if line.strip()
]
golden = {
    row['id']: row 
    for row in (
        json.loads(line) 
        for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() 
        if line.strip()
    )
}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [3]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. One sentence, no examples, no persona."""
    return [
        {
            "role": "user",
            "content": (
                "Extract the following fields from the job posting below and return them "
                "as a JSON object with exactly these keys: "
                '"company", "role", "years_experience_required".\n'
                "Use null for years_experience_required if no specific number is stated.\n\n"
                f"Job posting:\n{snippet_text}\n\n"
                "Return only the JSON object, no extra text."
            ),
        }
    ]

FEW_SHOT_EXAMPLES = """
Example 1:
Posting: "TechCorp is looking for a Backend Engineer with at least 3 years of Python experience."
Output: {"company": "TechCorp", "role": "Backend Engineer", "years_experience_required": 3}

Example 2:
Posting: "BrightHealth needs a Data Scientist. Fresh graduates welcome — no experience required."
Output: {"company": "BrightHealth", "role": "Data Scientist", "years_experience_required": 0}

Example 3:
Posting: "We're hiring. The role is a Product Lead. We're a fast-moving team and don't specify a years requirement."
Output: {"company": null, "role": "Product Lead", "years_experience_required": null}
""".strip()


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Three worked examples before the actual snippet."""
    return [
        {
            "role": "user",
            "content": (
                "Extract three fields from a job posting: "
                '"company", "role", and "years_experience_required".\n'
                "Return a JSON object with exactly those keys.\n"
                "Use null for years_experience_required if no specific number is stated.\n"
                "Use the integer value (e.g. 5, not '5+') for years.\n\n"
                f"{FEW_SHOT_EXAMPLES}\n\n"
                f"Now extract from this posting:\n{snippet_text}\n\n"
                "Output:"
            ),
        }
    ]


STRUCTURED_SYSTEM = (
    "You are an expert recruiter and data extraction specialist. "
    "Your job is to parse job posting text and return structured data. "
    "You always return valid JSON — nothing else. "
    "You are precise: you never infer or fabricate data that is not explicitly stated."
)

STRUCTURED_SCHEMA = (
    "Return a JSON object with exactly these fields:\n"
    "  company (string | null)                — the name of the hiring company\n"
    "  role (string | null)                   — the exact job title\n"
    "  years_experience_required (int | null)  — the minimum years of experience as an integer;\n"
    "                                            use the lower bound of a range (e.g. '3-5 years' → 3);\n"
    "                                            strip '+' and use the number (e.g. '7+' → 7);\n"
    "                                            use 0 if the posting says no experience is required;\n"
    "                                            use null ONLY if no years figure is mentioned at all.\n"
    "Do not include any explanation, markdown, or text outside the JSON object."
)


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured/role-based. System persona + explicit JSON schema."""
    return [
        {"role": "system", "content": STRUCTURED_SYSTEM},
        {
            "role": "user",
            "content": (
                f"{STRUCTURED_SCHEMA}\n\n"
                f"Job posting:\n{snippet_text}"
            ),
        },
    ]


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Reason step by step, then emit JSON."""
    return [
        {
            "role": "user",
            "content": (
                "Read the job posting below and extract three fields: "
                '"company", "role", and "years_experience_required".\n\n'
                "Think step by step:\n"
                "  1. Identify the company name. If it is not stated, note that.\n"
                "  2. Identify the job title / role.\n"
                "  3. Find any mention of years of experience. "
                "If a range is given, take the lower bound. "
                "If '0' or 'no experience required' is stated, use 0. "
                "If nothing is mentioned, the value is null.\n"
                "  4. Write your final answer as a JSON object with keys "
                '"company", "role", "years_experience_required".\n\n'
                f"Job posting:\n{snippet_text}\n\n"
                "Work through the steps above, then end with the JSON object."
            ),
        }
    ]


STRATEGIES = {
    "zero_shot": prompt_zero_shot,
    "few_shot": prompt_few_shot,
    "structured": prompt_structured,
    "cot": prompt_cot,
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [5]:
def parse_response(text: str) -> dict | None:
    """
    Extract and parse a JSON object from a model response.

    Handles:
      • Pure JSON string
      • JSON wrapped in ```json ... ``` or ``` ... ``` fences
      • JSON embedded after chain-of-thought prose
    """
    if not text or not text.strip():
        return None

    # Strip markdown code fences
    cleaned = re.sub(r'```(?:json)?\s*', '', text)
    cleaned = cleaned.replace('```', '')

    # Try direct parse first (handles pure JSON or fenced JSON)
    try:
        return json.loads(cleaned.strip())
    except json.JSONDecodeError:
        pass

    # Find the last JSON object in the text (handles CoT preamble)
    # Use last match so we skip any reasoning and grab the final answer
    matches = list(re.finditer(r'\{[^{}]*\}', cleaned, re.DOTALL))
    if matches:
        try:
            return json.loads(matches[-1].group())
        except json.JSONDecodeError:
            pass

    return None


def compute_cost(usage, model: str) -> float:
    """Compute USD cost from a usage object and model name."""
    rates = RATES.get(model, RATES['gpt-4o-mini'])
    return (
        usage.prompt_tokens     * rates['in'] +
        usage.completion_tokens * rates['out']
    )


print('Helper functions defined.')


async def run_one(strategy_name: str, snippet: dict) -> dict:
    """
    Run one strategy against one snippet.

    Returns a dict with:
      strategy, snippet_id, raw_response, parsed, cost_usd, latency_s
    """
    prompt_fn = STRATEGIES[strategy_name]
    messages  = prompt_fn(snippet['snippet'])

    t0 = time.perf_counter()
    response = await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=TEMPERATURE,
    )
    latency_s = time.perf_counter() - t0

    raw_text = response.choices[0].message.content or ''
    parsed   = parse_response(raw_text)
    cost_usd = compute_cost(response.usage, MODEL)

    return {
        'strategy':     strategy_name,
        'snippet_id':   snippet['id'],
        'raw_response': raw_text,
        'parsed':       parsed,
        'cost_usd':     cost_usd,
        'latency_s':    latency_s,
    }


async def run_all() -> list[dict]:
    """
    Run all 10 × 4 = 40 calls concurrently using asyncio.gather.
    Returns a flat list of result dicts.
    """
    tasks = [
        run_one(strategy_name, snippet)
        for strategy_name in STRATEGIES
        for snippet in snippets
    ]
    print(f'Launching {len(tasks)} concurrent LLM calls …')
    results = await asyncio.gather(*tasks)
    print(f'Done. Got {len(results)} results.')
    return list(results)


print('Async functions defined.')

Helper functions defined.
Async functions defined.


In [6]:
# ── Run all 40 calls (or load from cache) ───────────────────────────────────
RAW_CACHE = Path('../results_raw.json')

if RAW_CACHE.exists():
    print(f'Loading cached raw results from {RAW_CACHE}')
    results = json.loads(RAW_CACHE.read_text(encoding='utf-8'))
else:
    # In Jupyter use 'await' directly — do NOT wrap in asyncio.run()
    results = await run_all()
    RAW_CACHE.write_text(json.dumps(results, indent=2), encoding='utf-8')
    print(f'Raw results cached to {RAW_CACHE}')

print(f'Got {len(results)} results.')
results[0]

Loading cached raw results from ..\results_raw.json
Got 40 results.


{'strategy': 'zero_shot',
 'snippet_id': 'j01',
 'raw_response': '```json\n{\n  "company": "Acme Corp",\n  "role": "Senior Software Engineer",\n  "years_experience_required": 5\n}\n```',
 'parsed': {'company': 'Acme Corp',
  'role': 'Senior Software Engineer',
  'years_experience_required': 5},
 'cost_usd': 3.585e-05,
 'latency_s': 2.4120605997741222}

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [7]:
def normalise_str(value) -> str:
    """Lowercase, strip whitespace from a string field."""
    if value is None:
        return ''
    return str(value).strip().lower()


def normalise_years(value) -> int | None:
    """
    Coerce years_experience_required to int or None.
    Handles: int, float, '5', '5+', '3-5' (→ 3), null/None.
    """
    if value is None:
        return None
    s = str(value).strip()
    if s.lower() in ('null', 'none', ''):
        return None
    s = s.strip('+')
    m = re.match(r'(\d+)', s)
    if m:
        return int(m.group(1))
    return None


def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """
    Compare three fields against the golden record.
    Returns 0–3 (one point per correct field).

    Rules:
      company / role  — case-insensitive string match after whitespace trim
      years           — integer match; None must match null in golden
    """
    if extracted is None:
        return 0

    score = 0

    # company
    if normalise_str(extracted.get('company')) == normalise_str(gold.get('company')):
        score += 1

    # role
    if normalise_str(extracted.get('role')) == normalise_str(gold.get('role')):
        score += 1

    # years_experience_required
    extracted_years = normalise_years(extracted.get('years_experience_required'))
    gold_years      = gold.get('years_experience_required')  # already int or None
    if extracted_years == gold_years:
        score += 1

    return score


print('Scoring functions defined.')

Scoring functions defined.


In [8]:
# LLM-as-judge rubric — holistic 1–25 score
JUDGE_SYSTEM = (
    'You are an evaluation judge assessing how accurately an AI model extracted '
    'structured data from a job posting. You will be given the source text, the '
    'reference (correct) extraction, and the model\'s extraction. '
    'Score the model\'s extraction from 1 to 25 using this rubric:\n\n'
    '  21–25 — All three fields correct, exact or near-exact match.\n'
    '  15–20 — Two of three fields correct, no fabricated data.\n'
    '   8–14 — One field correct, or minor field errors.\n'
    '   1– 7 — No fields correct, response unparsable, or significant hallucination.\n\n'
    'Penalise heavily if the model fabricates a years figure when the posting does not state one.\n'
    'Reply with a single integer and nothing else.'
)


async def score_llm_judge(
    snippet_text: str,
    extracted: dict | None,
    gold: dict,
) -> int:
    """
    Ask gpt-4o to judge how well the extraction matches the golden record.
    Returns an integer 1–25.
    """
    user_msg = (
        f'Source text:\n{snippet_text}\n\n'
        f'Reference extraction:\n'
        f'{json.dumps({k: gold[k] for k in ("company", "role", "years_experience_required")}, indent=2)}\n\n'
        f'Model extraction:\n'
        f'{json.dumps(extracted, indent=2) if extracted else "null (failed to parse)"}\n\n'
        'Score (1–25):'
    )

    try:
        response = await client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[
                {'role': 'system', 'content': JUDGE_SYSTEM},
                {'role': 'user',   'content': user_msg},
            ],
            temperature=0.0,
        )
        raw = response.choices[0].message.content.strip()
        return int(re.search(r'\d+', raw).group())
    except Exception as e:
        print(f'  Judge error: {e}')
        return 1


print('LLM judge function defined.')

LLM judge function defined.


In [9]:
# ── Apply scoring to all 40 results ─────────────────────────────────────────
SCORED_CACHE = Path('../results_scored.json')

if SCORED_CACHE.exists():
    print(f'Loading cached scored results from {SCORED_CACHE}')
    scored = json.loads(SCORED_CACHE.read_text(encoding='utf-8'))
else:
    # Build snippet text lookup
    snippet_lookup = {s['id']: s['snippet'] for s in snippets}

    # Step 4a — deterministic scores (no API call)
    for row in results:
        gold = golden[row['snippet_id']]
        row['parse_success'] = row['parsed'] is not None
        row['accuracy']      = score_accuracy(row['parsed'], gold)

    # Step 4b — LLM judge (fire all 40 concurrently)
    print('Running LLM judge on all 40 results …')
    judge_tasks = [
        score_llm_judge(
            snippet_lookup[row['snippet_id']],
            row['parsed'],
            golden[row['snippet_id']],
        )
        for row in results
    ]
    judge_scores = await asyncio.gather(*judge_tasks)

    for row, js in zip(results, judge_scores):
        row['llm_judge_score'] = js

    scored = results
    SCORED_CACHE.write_text(json.dumps(scored, indent=2), encoding='utf-8')
    print(f'Scored results cached to {SCORED_CACHE}')

print(f'Scored {len(scored)} results.')

Loading cached scored results from ..\results_scored.json
Scored 40 results.


## Step 5 — Build the comparison table

In [10]:
df = pd.DataFrame(scored)

summary = df.groupby('strategy').agg(
    accuracy        =('accuracy',        'mean'),
    parse_rate      =('parse_success',   'mean'),
    judge_score     =('llm_judge_score', 'mean'),
    total_cost_usd  =('cost_usd',        'sum'),
    latency_p50_s   =('latency_s',       'median'),
).round(3)

# Re-order rows in logical display order
_order = ['zero_shot', 'few_shot', 'structured', 'cot']
summary = summary.reindex([s for s in _order if s in summary.index])
summary.index = ['Zero-shot', 'Few-shot', 'Structured', 'Chain-of-thought']
summary.columns = ['Accuracy (mean/3)', 'Parse rate', 'Judge score',
                   'Cost ($)', 'Latency p50 (s)']

summary

,Accuracy (mean/3),Parse rate,Judge score,Cost ($),Latency p50 (s)
Zero-shot,2.7,1.0,24.2,0.000,2.754
Few-shot,2.8,1.0,24.6,0.001,2.331
Structured,2.9,1.0,25.0,0.001,2.436
Chain-of-thought,2.8,1.0,25.0,0.001,2.523


In [11]:
# ── Cost summary ─────────────────────────────────────────────────────────────
total_main  = sum(r['cost_usd'] for r in scored)
n_judge     = len(scored)
est_judge   = n_judge * (RATES['gpt-4o']['in'] * 300 + RATES['gpt-4o']['out'] * 10)

print(f'Main-strategy spend  (gpt-4o-mini, 40 calls): ${total_main:.5f}')
print(f'Est. judge spend     (gpt-4o,       40 calls): ~${est_judge:.5f}')
print(f'Estimated total:                               ~${total_main + est_judge:.5f}')

Main-strategy spend  (gpt-4o-mini, 40 calls): $0.00232
Est. judge spend     (gpt-4o,       40 calls): ~$0.03400
Estimated total:                               ~$0.03632


## Step 6 — Reflection

See [`../mp1_writeup.md`](../mp1_writeup.md) for the 1-page writeup answering:

1. Which strategy won, and on what dimension?
2. What surprised you?
3. Which strategy would you reach for first in your capstone domain?
4. If you had another day, what would you try next?

---

```bash
git add src/ data/ mp1_comparison.md mp1_writeup.md
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```